# שיעור 03 - דפוסי עיצוב סוכניים

בשיעור זה, נחקור שלושה דפוסי עיצוב יסודיים לבניית סוכני AI יעילים:

1. **הוראות סוכנות ברורות** — יצירת הנחיות מדויקות שמגדירות תפקיד ומנחות את התנהגות הסוכן  
2. **פלט מובנה עם מודלים של Pydantic** — הבטחת החזרת נתונים צפוים ומאומתים על ידי הסוכנים  
3. **סוכנים עם אחריות יחידה** — עיצוב סוכנים ממוקדים שכל אחד עושה דבר אחד היטב

ניישם כל דפוס על תרחיש של **ממליץ ליעדי טיול**, ונבנה בהדרגה מערכת שיכולה להציע יעדים, לבדוק זמינות, ולנהל לוגיסטיקה.


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## תבנית 1: הוראות ברורות לסוכן

התבנית בעלת ההשפעה הרבה ביותר היא גם הפשוטה ביותר: כתיבת הוראות ברורות ומפורטות לסוכן שלך.

הוראות טובות מגדירות:
- **מי** הסוכן (אישיות וטון)
- **מה** עליו לעשות (אחריות שלב אחר שלב)
- **איך** עליו להתנהל (אילוצים וסגנון)

להלן, אנו יוצרים סוכן קונסיירז' נסיעות עם הוראות מפורשות שמעצבות כל תגובה שהוא מייצר.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## תבנית 2: פלט מובנה עם מודלים של Pydantic

טקסט חופשי מועיל לשיחה, אבל מערכות בהמשך זקוקות לנתונים מובנים.
על ידי שילוב של **מודלים של Pydantic** עם **פונקציית כלי**, אנו יכולים:

- להגדיר סכימה מדויקת לפלט של הסוכן
- לאמת תגובות באופן אוטומטי
- לשלב את תוצאות הסוכן בלוגיקת היישום בצורה מהימנה

אנו גם מציגים כלי שמחזיר פרטי יעד כך שהסוכן יבסס את ההמלצות שלו על נתונים אמיתיים.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## תבנית 3: סוכנים עם אחריות בודדת

משימות מורכבות נהנות מחלוקת העבודה בין כמה סוכנים ממוקדים, שכל אחד מהם אחראי על אחריות בודדת:

- **מומחה יעד** שיודע על מקומות וזמינות
- **מתכנן לוגיסטיקה** שמטפל בטיסות, מלונות, ונסיעות

זה משקף את עקרון ההנדסה התוכנה של *הפרדת דאגות* — כל סוכן קל יותר לבדוק, לתחזק ולשפר באופן עצמאי.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## סיכום

בשיעור זה יישמנו שלושה תבניות עיצוב סוכנייות בתרחיש של מציע טיולים:

| תבנית | רעיון מרכזי | יתרון |
|---|---|---|
| **הוראות ברורות** | הגדרת פרסונה, אחריות ומגבלות מראש | התנהגות סוכן עקבית ובהלימה למותג |
| **פלט מובנה** | שימוש בדגמי Pydantic כפורמט התגובה | תוצאות מאומתות, הניתנות לקריאה על ידי מכונה |
| **אחריות יחידה** | הענקת משימה ממוקדת לכל סוכן | קל יותר לבדוק, לתחזק ולרכוב |

תבניות אלו מתמזגות באופן טבעי — ניתן לשלב הוראות ברורות עם פלט מובנה בתוך סוכן בעל אחריות יחידה כדי לבנות מערכות איתנות ומוכנות לייצור.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
